## Notebook 17 — Kontrol Shuffled-Donor untuk Efek Merusak L11 H16 (review B6)

Finding 08: patch **satu head L11 H16** (donor = sel B) di RACExRELIG
menggeser prediksi MENJAUH dari kebenaran B — konsisten di hampir semua
pasangan (t=-5.2, p<1/4096). Dua bacaan yang belum terpisahkan:

1. **Korupsi generik** — menimpa 128 dimensi mid-computation dengan
   aktivasi konteks lain memang merusak, siapa pun donornya.
2. **Bahaya identitas-spesifik** — yang merusak adalah *konten identitas*
   donornya.

Kontrol di sini (semua patch di prompt A, head H16 saja, alpha=1, posisi
token identitas — protokol PERSIS notebook 12, seed & pasangan sama):

| kondisi | donor |
|---|---|
| `true` | aktivasi sel B (replikasi finding 08) |
| `othercell_1/_2` | aktivasi sel C acak (C bukan A, bukan B) setipe |
| `dimshuffle` | aktivasi B yang 128 dimensi head-nya dipermutasi (statistik sama, struktur hancur) |

Cara baca: kalau `othercell`/`dimshuffle` sama merusaknya dengan `true`
-> korupsi generik (klaim paper dipersempit); kalau `true` jelas lebih
merusak -> ada komponen identitas-spesifik.

Tipe: **RACExRELIG** (efeknya di sini) + **AGExPOLPARTY** (kontras, H16 n.s.).

Setting: GPU T4 x2, attach `opinionqa_intersectional.csv`, Internet On.
~2.400 forward pass, ±45-60 menit.


In [ ]:
!pip install -q -U "transformers>=4.44" accelerate scipy tqdm

In [ ]:
import os, sys, gc, glob, ast
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import numpy as np
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from scipy.stats import wasserstein_distance
from tqdm.auto import tqdm

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    for d in range(torch.cuda.device_count()):
        free, total = torch.cuda.mem_get_info(d)
        print(f"GPU {d}: {free/1e9:.1f} GB free / {total/1e9:.1f} GB total")
        if free / total < 0.9:
            print(f"  PERINGATAN: GPU {d} tidak kosong -> RESTART SESSION dulu!")


In [ ]:
MODEL_PATH = "mistralai/Mistral-7B-v0.1"

_candidates = glob.glob("/kaggle/input/**/opinionqa_intersectional.csv", recursive=True)
if _candidates:
    DATA_PATH = _candidates[0]
elif os.path.exists("opinionqa_intersectional.csv"):
    DATA_PATH = "opinionqa_intersectional.csv"
else:
    raise FileNotFoundError("opinionqa_intersectional.csv tidak ketemu.")
print("Data:", DATA_PATH)

RANDOM_SEED = 42
# PENTING: plan dibangun utk KETIGA tipe notebook 12 dgn urutan sama, supaya
# konsumsi rng identik -> pasangan RACExRELIG & AGExPOLPARTY sama persis
# dengan finding 08 (sebanding apple-to-apple). RELIGxPP di-skip saat patch.
PLAN_TYPES = ["AGExPOLPARTY", "RELIGxPOLPARTY", "RACExRELIG"]
TYPES_RUN = ["RACExRELIG", "AGExPOLPARTY"]
N_PAIRS = 12
N_QUESTIONS = 20
MAX_OPTIONS = 6
MIN_SHARED_Q = 20
STAR_LAYER, STAR_HEAD = 11, 16
N_OTHERCELL = 2

OUT_DIR = "/kaggle/working/shuffled_donor"
os.makedirs(OUT_DIR, exist_ok=True)


In [ ]:
df = pd.read_csv(DATA_PATH)
for c in ["responses", "ordinal", "options"]:
    df[c] = df[c].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
df["group_key"] = df["attribute"] + " :: " + df["group"]
df["n_opt"] = df["ordinal"].apply(len)

qmeta = {}
real_resp = {}
for r in df.itertuples():
    qmeta[r.qkey] = (r.question, r.options[: r.n_opt], r.ordinal)
    real_resp[(r.group_key, r.qkey)] = np.array(r.responses, dtype=np.float64)

def real_wd(A, B, qk):
    _, _, ordinal = qmeta[qk]
    return wasserstein_distance(ordinal, ordinal,
                                u_weights=real_resp[(A, qk)], v_weights=real_resp[(B, qk)])

rng = np.random.default_rng(RANDOM_SEED)
plan = {}
for ty in PLAN_TYPES:  # urutan & rng PERSIS notebook 12
    sub = df[(df["attribute"] == ty) & (df["n_opt"] <= MAX_OPTIONS)]
    cells = sorted(sub["group_key"].unique().tolist())
    q_per_cell = sub.groupby("group_key")["qkey"].apply(set).to_dict()
    v1_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[0] for gk in cells})
    v2_opts = sorted({gk.split(" :: ", 1)[1].split(" | ", 1)[1] for gk in cells})
    all_pairs = [(a, b) for a in cells for b in cells
                 if a != b and len(q_per_cell[a] & q_per_cell[b]) >= MIN_SHARED_Q]
    pick = rng.choice(len(all_pairs), size=min(N_PAIRS, len(all_pairs)), replace=False)
    pairs = [all_pairs[k] for k in pick]
    pair_questions = {}
    for (a, b) in pairs:
        shared = sorted(q_per_cell[a] & q_per_cell[b])
        wds = sorted(((real_wd(a, b, qk), qk) for qk in shared), reverse=True)
        pair_questions[(a, b)] = [qk for _, qk in wds[:N_QUESTIONS]]
    needed = sorted({(gk, qk) for (a, b), qs in pair_questions.items()
                     for qk in qs for gk in (a, b)})
    plan[ty] = dict(cells=sorted({c for p in pairs for c in p}), pairs=pairs,
                    pair_questions=pair_questions, needed=needed,
                    v1_opts=v1_opts, v2_opts=v2_opts,
                    all_cells=cells, q_per_cell=q_per_cell)
    print(f"[{ty}] {len(plan[ty]['cells'])} sel, {len(pairs)} pasangan, "
          f"baseline unik {len(needed)}")

# donor othercell: 2 sel acak per pasangan, C bukan A/B, deterministik
rng_shuf = np.random.default_rng(RANDOM_SEED + 17)
other_cells = {}
for ty in TYPES_RUN:
    p = plan[ty]
    for (a, b) in p["pairs"]:
        cand = [c for c in p["all_cells"] if c not in (a, b)
                and len(p["q_per_cell"][c]) > 0]
        pick = rng_shuf.choice(len(cand), size=N_OTHERCELL, replace=False)
        other_cells[(ty, a, b)] = [cand[k] for k in pick]
print("contoh othercell:", next(iter(other_cells.items())))


In [ ]:
ATTR_QA = {
    "RACExRELIG":       ("What is this survey respondent's race?",
                         "What is this survey respondent's religion?"),
    "AGExPOLPARTY":     ("What is this survey respondent's age group?",
                         "What is this survey respondent's political party affiliation?"),
}
DEMO_LETTERS = [chr(65 + i) for i in range(26)]
LETTERS = ["A", "B", "C", "D", "E", "F"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def demo_block(question, opts, value):
    lines = [f"Question: {question}"]
    for i, o in enumerate(opts):
        lines.append(f"{DEMO_LETTERS[i]}. {o}")
    lines.append(f"Answer: {DEMO_LETTERS[opts.index(value)]}")
    return "\n".join(lines)

def build_prompt(ty, gk, qk):
    v1, v2 = gk.split(" :: ", 1)[1].split(" | ", 1)
    q1, q2 = ATTR_QA[ty]
    p = plan[ty]
    question, options, _ = qmeta[qk]
    blocks = [demo_block(q1, p["v1_opts"], v1), "", demo_block(q2, p["v2_opts"], v2), "",
              f"Question: {question}"]
    for i, opt in enumerate(options):
        blocks.append(f"{LETTERS[i]}) {opt}")
    blocks.append("Answer:")
    return "\n".join(blocks)

def identity_positions(prompt):
    positions = []
    search_from = 0
    for _ in range(2):
        idx = prompt.find("Answer:", search_from)
        assert idx != -1
        prefix = prompt[: idx + len("Answer:")]
        pos = 1 + len(tokenizer.encode(prefix, add_special_tokens=False))  # +1 BOS
        positions.append(pos)
        search_from = idx + 1
    return positions

# validasi ala notebook 12
ty0 = TYPES_RUN[0]
(A0, B0) = plan[ty0]["pairs"][0]
qk0 = plan[ty0]["pair_questions"][(A0, B0)][0]
pA, pB = build_prompt(ty0, A0, qk0), build_prompt(ty0, B0, qk0)
tA = tokenizer(pA, return_tensors="pt")["input_ids"][0]
tB = tokenizer(pB, return_tensors="pt")["input_ids"][0]
posA = identity_positions(pA)
assert len(tA) == len(tB) and posA == identity_positions(pB)
diff = (tA != tB).nonzero().flatten().tolist()
assert set(diff).issubset(set(posA)), "token beda bukan di posisi identitas!"
print("OK. posisi identitas:", posA, "| token beda:", diff)


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH, torch_dtype=torch.float16, device_map="balanced", low_cpu_mem_usage=True
)
model.eval()
NUM_HEADS = model.config.num_attention_heads
HEAD_DIM = model.config.hidden_size // NUM_HEADS
S16 = slice(STAR_HEAD * HEAD_DIM, (STAR_HEAD + 1) * HEAD_DIM)

LETTER_IDS = [tokenizer.encode(f" {L}", add_special_tokens=False)[-1] for L in LETTERS]
assert len(set(LETTER_IDS)) == len(LETTER_IDS)

_donor_capture = {}
_capture_positions = []
_active_patch = {}

def _oproj_prehook(layer_idx):
    def fn(module, args):
        x = args[0]
        if _capture_positions:
            _donor_capture[layer_idx] = {
                pos: x[:, pos, :].detach().float().cpu() for pos in _capture_positions
            }
        patches = _active_patch.get(layer_idx)
        if patches:
            x = x.clone()
            for (pos, heads, alpha, donor) in patches:
                d = donor.to(x.device, x.dtype)
                for h in heads:
                    s = slice(h * HEAD_DIM, (h + 1) * HEAD_DIM)
                    x[0, pos, s] = x[0, pos, s] + alpha * (d[s] - x[0, pos, s])
            return (x,) + tuple(args[1:])
        return None
    return fn

handle = model.model.layers[STAR_LAYER].self_attn.o_proj.register_forward_pre_hook(
    _oproj_prehook(STAR_LAYER))
print("Hook di layer", STAR_LAYER)

@torch.no_grad()
def forward_pred(prompt, n_opt, capture_positions=None, patch_spec=None):
    global _capture_positions
    _capture_positions = capture_positions or []
    _active_patch.clear()
    if patch_spec:
        _active_patch.update(patch_spec)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    logits = model(**inputs).logits[0, -1, :]
    _active_patch.clear()
    _capture_positions = []
    sel = logits[LETTER_IDS[:n_opt]].float()
    return torch.softmax(sel, dim=0).cpu().numpy()


## Pass 1 — baseline pred + donor per (sel, soal)

Donor ditangkap **per (sel, soal)** — protokol persis notebook 12, tanpa
jalan pintas. Secara matematis aktivasi di posisi token identitas tidak
tergantung soal (attention kausal: soal opini ada SESUDAH posisi itu),
tapi secara numerik kernel fp16 memilih jalur/tiling berbeda untuk
panjang sekuens berbeda, jadi nilainya meleset di digit terakhir.
Daripada mengandalkan asumsi itu, kita tangkap apa adanya — sekalian
bikin kondisi `true` sebanding penuh dengan finding 08.

Sel donor `othercell` juga ditangkap pada soal yang sama. Sel itu tidak
perlu punya data survei untuk soal tersebut: yang dibutuhkan cuma
aktivasinya, bukan jawaban aslinya.


In [ ]:
donors = {}         # (ty, gk, qk) -> {pos: vec4096}
baseline_pred = {}  # (gk, qk) -> pred
id_pos_type = {}    # ty -> positions (konstan per tipe, dicek di bawah)

def capture(ty, gk, qk, want_pred):
    pr = build_prompt(ty, gk, qk)
    positions = identity_positions(pr)
    id_pos_type.setdefault(ty, positions)
    assert id_pos_type[ty] == positions, (ty, gk, qk, positions)
    pred = forward_pred(pr, len(qmeta[qk][2]), capture_positions=positions)
    donors[(ty, gk, qk)] = {pos: _donor_capture[STAR_LAYER][pos][0].clone()
                            for pos in positions}
    if want_pred:
        baseline_pred[(gk, qk)] = pred

# 1. sel A & B di semua soal pasangan -> baseline pred + donor
for ty in TYPES_RUN:
    for (gk, qk) in tqdm(plan[ty]["needed"], desc=f"baseline+donor {ty}"):
        if (ty, gk, qk) not in donors:
            capture(ty, gk, qk, want_pred=True)

# 2. sel donor othercell di soal yang sama (cuma butuh aktivasi)
for ty in TYPES_RUN:
    p = plan[ty]
    todo = sorted({(c, qk) for (a, b) in p["pairs"]
                   for c in other_cells[(ty, a, b)]
                   for qk in p["pair_questions"][(a, b)]})
    for (gk, qk) in tqdm(todo, desc=f"donor othercell {ty}"):
        if (ty, gk, qk) not in donors:
            capture(ty, gk, qk, want_pred=False)

print(f"{len(baseline_pred)} baseline, {len(donors)} donor selesai.")

# Diagnostik (BUKAN assert): beda donor sel yang sama antar 2 soal.
# Harusnya ~0 secara matematis; sisa beda = noise numerik fp16. Kalau cos
# jauh dari 1.0 (mis. <0.99), baru ada yang salah dengan posisi identitas.
ty0 = TYPES_RUN[0]
gk0 = plan[ty0]["cells"][0]
qs0 = sorted({qk for (t, g, qk) in donors if t == ty0 and g == gk0})[:2]
if len(qs0) == 2:
    print(f"\ncek numerik [{gk0}] soal {qs0[0]} vs {qs0[1]}:")
    for pos in id_pos_type[ty0]:
        a, b = donors[(ty0, gk0, qs0[0])][pos], donors[(ty0, gk0, qs0[1])][pos]
        cos = float(torch.nn.functional.cosine_similarity(a, b, dim=0))
        print(f"  pos {pos}: max|beda|={float((a - b).abs().max()):.2e} "
              f"| |v|={float(a.norm()):.1f} | cos={cos:.6f}")


## Pass 2 — 4 kondisi patch H16 di prompt A

`true` = donor B (replikasi finding 08) | `othercell_1/_2` = donor sel C
acak | `dimshuffle` = donor B dengan 128 dim head-16 dipermutasi
(permutasi tetap per pasangan, seed 42+23).


In [ ]:
rng_perm = np.random.default_rng(RANDOM_SEED + 23)

def wd(pred, real, ordinal):
    return wasserstein_distance(ordinal, ordinal, u_weights=pred, v_weights=real)

def spec_from(ty, donor_map, alpha=1.0):
    return {STAR_LAYER: [(pos, [STAR_HEAD], alpha, donor_map[pos])
                         for pos in id_pos_type[ty]]}

rows = []
for ty in TYPES_RUN:
    p = plan[ty]
    for pi, (A, B) in enumerate(tqdm(p["pairs"], desc=f"patch {ty}")):
        oc = other_cells[(ty, A, B)]
        # permutasi dimensi TETAP per pasangan (semua soal pakai yang sama)
        perm = torch.from_numpy(rng_perm.permutation(HEAD_DIM)).long()

        for qk in p["pair_questions"][(A, B)]:
            question, options, ordinal = qmeta[qk]
            n_opt = len(ordinal)
            realA, realB = real_resp[(A, qk)], real_resp[(B, qk)]
            predA, predB = baseline_pred[(A, qk)], baseline_pred[(B, qk)]
            prompt_A = build_prompt(ty, A, qk)

            donor_true = donors[(ty, B, qk)]
            donor_dimshuf = {}
            for pos, v in donor_true.items():
                v2 = v.clone()
                v2[S16] = v2[S16][perm]
                donor_dimshuf[pos] = v2

            conds = [("true", donor_true)]
            conds += [(f"othercell_{k+1}", donors[(ty, oc[k], qk)])
                      for k in range(N_OTHERCELL)]
            conds += [("dimshuffle", donor_dimshuf)]

            base = dict(attr_type=ty, pair=f"{A} -> {B}", qkey=qk,
                        wd_A_to_realA=wd(predA, realA, ordinal),
                        wd_A_to_realB=wd(predA, realB, ordinal),
                        wd_B_to_realB=wd(predB, realB, ordinal),
                        wd_predA_predB=wd(predA, predB, ordinal),
                        real_wd_AB=real_wd(A, B, qk))
            for cond, donor_map in conds:
                pp = forward_pred(prompt_A, n_opt,
                                  patch_spec=spec_from(ty, donor_map))
                rows.append(dict(base, condition=cond,
                                 donor_cells="|".join(oc) if cond.startswith("othercell") else "",
                                 wd_patch_to_realB=wd(pp, realB, ordinal),
                                 wd_patch_to_realA=wd(pp, realA, ordinal),
                                 wd_patch_to_predA=wd(pp, predA, ordinal),
                                 wd_patch_to_predB=wd(pp, predB, ordinal)))

res = pd.DataFrame(rows)
res["shift_ke_realB"] = res["wd_A_to_realB"] - res["wd_patch_to_realB"]
res["shift_ke_predB"] = res["wd_predA_predB"] - res["wd_patch_to_predB"]
res["gerak_total"] = res["wd_patch_to_predA"]  # seberapa jauh patch menggeser prediksi
res.to_csv(os.path.join(OUT_DIR, "shuffled_donor_rows.csv"), index=False)
print(res.shape, "-> shuffled_donor_rows.csv")


In [ ]:
print("=" * 100)
for ty in TYPES_RUN:
    r_ty = res[res["attr_type"] == ty]
    print(f"\n{ty}")
    for cond in ["true", "othercell_1", "othercell_2", "dimshuffle"]:
        s = r_ty[r_ty["condition"] == cond]
        print(f"  {cond:12s} shift->realB={s['shift_ke_realB'].mean():+.4f} "
              f"(>0: {(s['shift_ke_realB']>0).mean():.0%}) | "
              f"shift->predB={s['shift_ke_predB'].mean():+.4f} | "
              f"gerak total={s['gerak_total'].mean():.4f}")


## Download & analisis

Download `shuffled_donor/shuffled_donor_rows.csv` ->
`notebooks/output/17_shuffled_donor_kaggle/`, lalu:

```
./venv/Scripts/python.exe analisis_lokal/shuffled_donor.py
```

(script analisis dibuat setelah data ada): per tipe, level PASANGAN
(12 cluster, sign-flip eksak):
1. replikasi `true` vs finding 08 (harus t~-5.2 di RACExRELIG);
2. **uji kunci**: selisih per-pasangan `shift_true - shift_othercell`
   dan `shift_true - shift_dimshuffle` — kalau ~0, korupsi generik;
   kalau signifikan lebih negatif, ada komponen identitas-spesifik;
3. `gerak_total` antar kondisi (apakah semua kondisi menggeser prediksi
   sama jauhnya secara mekanis).

Hasil -> finding 14 + paper Sec. 6.3 butir 2 & Discussion butir 4.
